In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report
from model import GraphSAGE
from collections import deque
np.random.seed(42)

c:\Users\tusha\miniconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint = torch.load('models/task_a_model.pt', weights_only=False)
data = checkpoint['data']
concept_id_map = checkpoint['concept_id_map']
student_id_map = checkpoint['student_id_map']
course_id_map = checkpoint['course_id_map']

In [3]:
students = pd.read_csv('datasets/students.csv')
chatbot = pd.read_csv('datasets/chatbot_signals.csv')
course_concepts_df = pd.read_csv('datasets/course_concepts.csv')
concepts_df = pd.read_csv('datasets/concepts.csv')
enrollments = pd.read_csv('datasets/enrollments.csv')

concept_name_map = concepts_df.set_index('concept_id')['concept_name'].to_dict()
course_name_map = pd.read_csv('datasets/courses.csv').set_index('course_id')['course_name'].to_dict()
idx_to_concept = {v: k for k, v in concept_id_map.items()}

In [4]:
concept_prereqs = pd.read_csv('datasets/concept_prerequisites.csv')

real_edges = set()
for _, row in concept_prereqs.iterrows():
    src = concept_id_map[row['concept_id']]
    dst = concept_id_map[row['prereq_concept_id']]
    real_edges.add((src, dst))

pos_src = []
pos_dst = []
for _, row in concept_prereqs.iterrows():
    src = concept_id_map[row['concept_id']]
    dst = concept_id_map[row['prereq_concept_id']]
    pos_src.append(src)
    pos_dst.append(dst)

neg_src = []
neg_dst = []
for _ in range(len(pos_src) * 10):
    while True:
        i = np.random.randint(0, 80)
        j = np.random.randint(0, 80)
        if i != j and (i, j) not in real_edges:
            neg_src.append(i)
            neg_dst.append(j)
            break

print(f"Positive pairs: {len(pos_src)}")
print(f"Negative pairs: {len(neg_src)}")

Positive pairs: 89
Negative pairs: 890


In [5]:
all_src = pos_src + neg_src
all_dst = pos_dst + neg_dst
all_labels = [1] * len(pos_src) + [0]*len(neg_src)

src_tensor = torch.tensor(all_src, dtype = torch.long)
dst_tensor = torch.tensor(all_dst, dtype=torch.long)
labels = torch.tensor(all_labels, dtype = torch.float)

print(f"Total pairs: {len(all_labels)}")
print(f"Positives: {sum(all_labels)}, Negatives: {len(all_labels) - sum(all_labels)}")

np.random.seed(42)
indices = np.arange(len(all_labels))
np.random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx = indices[split:]

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

Total pairs: 979
Positives: 89, Negatives: 890
Train: 783, Test: 196


In [6]:
model = GraphSAGE(256)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
with torch.no_grad():
    out = model(data)
    student_embs = out[0]
    concept_embs = out[2]

print("Concept embeddings shape:", concept_embs.shape)
print("Student embeddings shape:", student_embs.shape)

Concept embeddings shape: torch.Size([80, 256])
Student embeddings shape: torch.Size([2000, 256])


In [7]:
class LinkPredictor(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, src_emb, dst_emb):
        pair = torch.concat([src_emb, dst_emb, src_emb - dst_emb, src_emb * dst_emb], dim = 1)
        return self.net(pair).squeeze()
    
link_model = LinkPredictor(emb_dim=256)
print(link_model)

LinkPredictor(
  (net): Sequential(
    (0): Linear(in_features=1024, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [8]:
optimizer = torch.optim.Adam(link_model.parameters(),lr = 0.001)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(1000):
    link_model.train()
    optimizer.zero_grad()

    train_src_emb = concept_embs[src_tensor[train_idx]]
    train_dst_emb = concept_embs[dst_tensor[train_idx]]
    train_labels = labels[train_idx]

    preds = link_model(train_src_emb, train_dst_emb)
    loss = loss_fn(preds, train_labels)

    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 0.7449
Epoch 200 | Loss: 0.1079
Epoch 400 | Loss: 0.0123
Epoch 600 | Loss: 0.0022
Epoch 800 | Loss: 0.0013


In [9]:
link_model.eval()
with torch.no_grad():
    test_src_emb = concept_embs[src_tensor[test_idx]]
    test_dst_emb = concept_embs[dst_tensor[test_idx]]
    test_labels = labels[test_idx]
    
    # Basic accuracy first
    preds = link_model(test_src_emb, test_dst_emb)
    pred_labels = (torch.sigmoid(preds) > 0.5).float()
    accuracy = (pred_labels == test_labels).float().mean()
    print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.8776


In [10]:
link_model.eval()
with torch.no_grad():
    hits_at_3 = 0
    mrr_sum = 0
    total = 0

    for _, row in concept_prereqs.iterrows():
        src = concept_id_map[row['concept_id']]
        true_dst = concept_id_map[row['prereq_concept_id']]

        src_emb = concept_embs[src].unsqueeze(0).repeat(80, 1)
        all_dst_emb = concept_embs

        scores = link_model(src_emb, all_dst_emb)
        scores = torch.sigmoid(scores)

        ranked = scores.argsort(descending=True)

        rank = (ranked == true_dst).nonzero(as_tuple = True)[0].item() + 1

        if rank <=3:
            hits_at_3 +=1
        mrr_sum += 1.0 /rank
        total += 1

    print(f"Hits@3: {hits_at_3/total:.4f} ({hits_at_3}/{total})")
    print(f"MRR: {mrr_sum/total:.4f}")

Hits@3: 0.8090 (72/89)
MRR: 0.6291


In [11]:
def get_prereq_chain(concept_ids, top_n=3):
    all_needed = set(concept_ids)
    
    link_model.eval()
    with torch.no_grad():
        for concept_id in concept_ids:
            c_idx = concept_id_map[concept_id]
            src_emb = concept_embs[c_idx].unsqueeze(0).repeat(80, 1)
            
            scores = torch.sigmoid(link_model(src_emb, concept_embs))
            
            # Don't link to yourself
            scores[c_idx] = 0.0
            
            # Get top_n highest scoring prereqs
            top_indices = scores.argsort(descending=True)[:top_n]
            
            for idx in top_indices:
                prereq_id = idx_to_concept[idx.item()]
                all_needed.add(prereq_id)
    
    return all_needed

In [12]:
def get_weak_concepts(student_id, course_id, top_k=3):
    # Step 1: Get concepts this course covers
    course_concepts = course_concepts_df[
        course_concepts_df['course_id'] == course_id
    ]['concept_id'].tolist()
    
    # Step 2: Expand prereqs using LinkPredictor
    all_needed = get_prereq_chain(course_concepts)
    
    # Step 3: Check student's chatbot confusion for these concepts
    student_signals = chatbot[chatbot['student_id'] == student_id]
    student_confusion = student_signals.set_index('concept_id')['confusion_score'].to_dict()
    
    # Step 4: Collect confusion scores
    gap_scores = []
    no_data = []
    for concept_id in all_needed:
        confusion = student_confusion.get(concept_id, None)
        if confusion is not None:
            gap_scores.append((concept_id, confusion))
        else:
            no_data.append((concept_id, -1))
    
    # Step 5: Rank by confusion, worst first
    gap_scores.sort(key=lambda x: x[1], reverse=True)
    return gap_scores[:top_k], no_data

In [13]:
def sequence_concepts(concept_ids):
    concept_list = list(concept_ids)
    concept_set = set(concept_ids)
    
    # Score all pairs within our set
    pair_scores = {}
    link_model.eval()
    with torch.no_grad():
        for concept_id in concept_list:
            c_idx = concept_id_map[concept_id]
            src_emb = concept_embs[c_idx].unsqueeze(0).repeat(80, 1)
            scores = torch.sigmoid(link_model(src_emb, concept_embs))
            
            for other_id in concept_list:
                if other_id == concept_id:
                    continue
                o_idx = concept_id_map[other_id]
                pair_scores[(concept_id, other_id)] = scores[o_idx].item()
    
    # Build DAG: if LinkPredictor says B is prereq of A with high score,
    # then B comes before A
    in_degree = {c: 0 for c in concept_set}
    adj = {c: [] for c in concept_set}
    
    for (concept_id, other_id), score in pair_scores.items():
        # concept_id says other_id is its prereq
        # so other_id should come BEFORE concept_id
        if score >= 0.7:
            adj[other_id].append(concept_id)
            in_degree[concept_id] += 1
    
    # Kahn's algorithm
    queue = deque([c for c in concept_set if in_degree[c] == 0])
    ordered = []
    
    while queue:
        current = queue.popleft()
        ordered.append(current)
        for child in adj[current]:
            in_degree[child] -= 1
            if in_degree[child] == 0:
                queue.append(child)
    
    # Handle any cycles
    if len(ordered) != len(concept_set):
        remaining = concept_set - set(ordered)
        ordered.extend(sorted(remaining))
    
    return ordered

In [14]:
def get_learning_path(student_id, course_id, top_k=3):
    course_name = course_name_map.get(course_id, course_id)
    
    # Step 1: Find weak concepts (uses LinkPredictor + chatbot data)
    weak, no_data = get_weak_concepts(student_id, course_id, top_k)
    
    if not weak:
        print(f"✓ {student_id} has no concept gaps for {course_name}")
        return []
    
    weak_ids = [c for c, _ in weak]
    weak_scores = {c: s for c, s in weak}
    
    # Step 2: Get prereqs for just the weak concepts
    all_needed = get_prereq_chain(weak_ids)
    
    # Step 3: Sequence into study order
    ordered = sequence_concepts(all_needed)
    
    # Step 4: Print the learning path
    print(f"\n{'='*60}")
    print(f"Learning Path: {student_id} → {course_name}")
    print(f"{'='*60}")
    print(f"Weak concepts: {len(weak_ids)}")
    print(f"Total to study (including prereqs): {len(ordered)}")
    print(f"{'-'*60}")
    
    path = []
    for i, concept_id in enumerate(ordered, 1):
        name = concept_name_map.get(concept_id, concept_id)
        
        if concept_id in weak_scores:
            tag = f"WEAK (confusion: {weak_scores[concept_id]:.3f})"
        else:
            tag = "PREREQ (foundation)"
        
        print(f"  {i}. [{concept_id}] {name} — {tag}")
        path.append((concept_id, name, tag))
    
    print(f"{'='*60}\n")
    if no_data:
        print(f"\n  Concepts never encountered ({len(no_data)}):")
        for concept_id, _ in no_data:
            name = concept_name_map.get(concept_id, concept_id)
            print(f"    - [{concept_id}] {name}")
    return path

In [15]:
high_risk = enrollments[enrollments['risk_class'] == 'High']
samples = high_risk.sample(3, random_state=42)

for _, row in samples.iterrows():
    path = get_learning_path(row['student_id'], row['course_id'], top_k=3)


Learning Path: S0097 → Data Pipelines
Weak concepts: 3
Total to study (including prereqs): 11
------------------------------------------------------------
  1. [K027] Statistical Modeling — PREREQ (foundation)
  2. [K071] User Research — PREREQ (foundation)
  3. [K031] Data Cleaning — WEAK (confusion: 0.653)
  4. [K050] TCP IP Stack — PREREQ (foundation)
  5. [K076] Color Theory — PREREQ (foundation)
  6. [K022] Calculus Fundamentals — PREREQ (foundation)
  7. [K001] Variables and Data Types — PREREQ (foundation)
  8. [K032] Exploratory Analysis — PREREQ (foundation)
  9. [K003] Functions and Recursion — WEAK (confusion: 0.427)
  10. [K042] Data Pipelines — WEAK (confusion: 0.551)
  11. [K069] ETL Processes — PREREQ (foundation)


  Concepts never encountered (7):
    - [K061] Relational Models
    - [K066] NoSQL Databases
    - [K068] Data Warehousing
    - [K062] SQL Queries
    - [K064] Indexing and Performance
    - [K063] Database Normalization
    - [K050] TCP IP Stack

Learning

In [16]:
torch.save(link_model.state_dict(), 'models/task_b_link_model_new.pt')
print("LinkPredictor saved!")

LinkPredictor saved!
